# Composite Transformer under Parallel Harmonics

CompositePowerComp did not support the harmonics domain, so a composite component (like the DP transformer, which is built from an inductor and several snubber elements) could not be simulated with `do_frequency_parallelization(True)`. This notebook builds a small circuit with a composite transformer, runs it in parallel-harmonics mode with two declared frequencies, and cross-checks the fundamental-frequency result against an independent single-frequency reference run of the same circuit.

In [ ]:
import villas.dataprocessing.readtools as rt
import matplotlib.pyplot as plt
import dpsimpy

## Parallel harmonics run

The source only excites the fundamental (50 Hz); the second declared frequency (250 Hz) carries no excitation, so its voltage should come out at exactly zero at both nodes. That is itself a useful check: it confirms the per-frequency solves are independent and not bleeding into each other.

In [ ]:
time_step = 0.0001
final_time = 0.01
sim_name = "DP_Transformer_Harmonics"
dpsimpy.Logger.set_log_dir("logs/" + sim_name)

frequencies = [50, 250]

n1 = dpsimpy.dp.SimNode("n1")
n2 = dpsimpy.dp.SimNode("n2")

log_level = dpsimpy.LogLevel.off

vs = dpsimpy.dp.ph1.VoltageSource("vs", log_level)
vs.set_parameters(complex(1000, 0))

trafo = dpsimpy.dp.ph1.Transformer("trafo", log_level)
trafo.set_parameters(
    nom_voltage_end_1=1000,
    nom_voltage_end_2=1000,
    rated_power=1e6,
    ratio_abs=1.0,
    ratio_phase=0.0,
    resistance=1.0,
    inductance=0.01,
)

load = dpsimpy.dp.ph1.Resistor("load", log_level)
load.set_parameters(100)

vs.connect([dpsimpy.dp.SimNode.gnd, n1])
trafo.connect([n1, n2])
load.connect([n2, dpsimpy.dp.SimNode.gnd])

sys = dpsimpy.SystemTopology(50, frequencies, [n1, n2], [vs, trafo, load])

logger = dpsimpy.Logger(sim_name)
logger.log_attribute("v1", "v", n1, 1, 2)
logger.log_attribute("v2", "v", n2, 1, 2)

sim = dpsimpy.Simulation(sim_name, log_level)
sim.set_system(sys)
sim.set_time_step(time_step)
sim.set_final_time(final_time)
sim.add_logger(logger)
sim.do_frequency_parallelization(True)

sim.run()

In [ ]:
log_path = "logs/" + sim_name + "/" + sim_name + ".csv"
ts_dpsim = rt.read_timeseries_dpsim(log_path)

In [ ]:
plt.plot(ts_dpsim["v1_0_0"].time, ts_dpsim["v1_0_0"].values, label="v1, 50 Hz")
plt.plot(ts_dpsim["v2_0_0"].time, ts_dpsim["v2_0_0"].values, label="v2, 50 Hz")
plt.xlabel("time (s)")
plt.ylabel("voltage phasor magnitude component (V)")
plt.legend()
plt.show()

In [ ]:
plt.plot(ts_dpsim["v1_0_1"].time, ts_dpsim["v1_0_1"].values, label="v1, 250 Hz")
plt.plot(ts_dpsim["v2_0_1"].time, ts_dpsim["v2_0_1"].values, label="v2, 250 Hz")
plt.xlabel("time (s)")
plt.ylabel("voltage phasor magnitude component (V)")
plt.legend()
plt.show()

In [ ]:
import numpy as np

assert np.allclose(ts_dpsim["v1_0_1"].values, 0.0)
assert np.allclose(ts_dpsim["v2_0_1"].values, 0.0)

## Reference: single-frequency sequential run of the same circuit

Same topology and parameters, run without harmonics at all, i.e. the well-tested path that composite components have always supported. The fundamental-frequency result above should match this exactly.

In [ ]:
sim_name_ref = "DP_Transformer_Harmonics_Reference"
dpsimpy.Logger.set_log_dir("logs/" + sim_name_ref)

n1_ref = dpsimpy.dp.SimNode("n1")
n2_ref = dpsimpy.dp.SimNode("n2")

vs_ref = dpsimpy.dp.ph1.VoltageSource("vs", log_level)
vs_ref.set_parameters(complex(1000, 0))

trafo_ref = dpsimpy.dp.ph1.Transformer("trafo", log_level)
trafo_ref.set_parameters(
    nom_voltage_end_1=1000,
    nom_voltage_end_2=1000,
    rated_power=1e6,
    ratio_abs=1.0,
    ratio_phase=0.0,
    resistance=1.0,
    inductance=0.01,
)

load_ref = dpsimpy.dp.ph1.Resistor("load", log_level)
load_ref.set_parameters(100)

vs_ref.connect([dpsimpy.dp.SimNode.gnd, n1_ref])
trafo_ref.connect([n1_ref, n2_ref])
load_ref.connect([n2_ref, dpsimpy.dp.SimNode.gnd])

sys_ref = dpsimpy.SystemTopology(50, [n1_ref, n2_ref], [vs_ref, trafo_ref, load_ref])

logger_ref = dpsimpy.Logger(sim_name_ref)
logger_ref.log_attribute("v1", "v", n1_ref)
logger_ref.log_attribute("v2", "v", n2_ref)

sim_ref = dpsimpy.Simulation(sim_name_ref, log_level)
sim_ref.set_system(sys_ref)
sim_ref.set_time_step(time_step)
sim_ref.set_final_time(final_time)
sim_ref.add_logger(logger_ref)

sim_ref.run()

In [ ]:
log_path_ref = "logs/" + sim_name_ref + "/" + sim_name_ref + ".csv"
ts_dpsim_ref = rt.read_timeseries_dpsim(log_path_ref)

In [ ]:
plt.plot(
    ts_dpsim["v2_0_0"].time, ts_dpsim["v2_0_0"].values, label="v2, parallel harmonics"
)
plt.plot(
    ts_dpsim_ref["v2"].time,
    ts_dpsim_ref["v2"].values,
    label="v2, sequential reference",
    linestyle="--",
)
plt.xlabel("time (s)")
plt.ylabel("voltage (V)")
plt.legend()
plt.show()

In [ ]:
assert np.allclose(ts_dpsim["v1_0_0"].values, ts_dpsim_ref["v1"].values)
assert np.allclose(ts_dpsim["v2_0_0"].values, ts_dpsim_ref["v2"].values)